In [1]:
%%capture
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [3]:
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU: Tesla T4


In [4]:
names = open("names.txt").read().splitlines()

In [5]:
len(names)

32033

In [6]:
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [7]:
chars = sorted(list(set(''.join(names))))
vocab_size = len(chars) + 1  # 0 = end token
chars


['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [8]:
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

def encode(name):
    return [stoi[c] for c in name] + [0]

def decode(indices):
    return ''.join(itos[i] for i in indices if i != 0)

In [10]:
block_size = 3
X = []
y = []

for name in names:
    encoded = encode(name)
    context = [0] * block_size
    for ch in encoded:
        X.append(context)
        y.append(ch)
        context = context[1:] + [ch]


X = torch.tensor(X)
y = torch.tensor(y)

X.shape, y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [11]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

n_embd = 10
n_hidden = 200

C = torch.randn((vocab_size, n_embd), device=device)
W1 = torch.randn((block_size*n_embd, n_hidden), device=device)
b1 = torch.zeros(n_hidden, device=device)
W2 = torch.randn((n_hidden, vocab_size), device=device)
b2 = torch.zeros(vocab_size, device=device)

params = [C, W1, b1, W2, b2]
for p in params:
    p.requires_grad = True

X = X.to(device)
y = y.to(device)


In [18]:
max_steps = 20000
lr = 0.1

for step in range(max_steps):

    # minibatch
    ix = torch.randint(0, X.shape[0], (256,), device=device)
    Xb = X[ix]
    Yb = y[ix]

    # forward
    emb = C[Xb]                      # (B, block, emb)
    h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in params:
        p.grad = None
    loss.backward()

    # gradient descent
    for p in params:
        p.data -= lr * p.grad

    if step % 2000 == 0:
        print(f"{step}: loss = {loss.item():.4f}")

print("Training done!")


0: loss = 2.0143
2000: loss = 2.2827
4000: loss = 2.1634
6000: loss = 2.0311
8000: loss = 2.1512
10000: loss = 2.0189
12000: loss = 2.1053
14000: loss = 2.1622
16000: loss = 1.8860
18000: loss = 2.1656
Training done!


In [19]:
def generate():
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context], device=device)]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        if ix == 0:
            break
        out.append(ix)
        context = context[1:] + [ix]
    return decode(out)

for _ in range(20):
    print(generate())


brie
nirabelle
elyn
lidi
izekslyn
annahanakobyn
vichaylane
elinett
kishente
aidelliana
aba
neiminori
serri
eymeyla
kenzie
wolfroda
noreeman
leg
khav
qura
